In [77]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "Croft96"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
try:
    with open(image_filename, 'rb') as image_file:
        encoded_image = base64.b64encode(image_file.read()).decode('utf-8')
        
    logo_element = html.Img(src=f'data:image/png;base64,{encoded_image}', style={'height': '100px', 'float': 'right'})
except FileNotFoundError:
    logo_element = html.H3("Graziosos Salvare Logo (Missing Link)", style={'float': 'right', 'color': 'red'})
    

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    logo_element,
    html.Center(html.B(html.H1("Grazioso Salvare - Animal Shelter Dashboard", style={'textAlign': 'left'}))),
    html.H4("Developed by: Samuel Leraas (ID: 3142998)", style={'textAlign': 'left', 'color': 'gray'}),

    html.Hr(),
    html.Div([
        html.Label("Select Rescue Type Filter:", style={'fontWeight': 'bold', 'fontSize': '16px'}),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': ' Water Rescue', 'value': 'Water'},
                {'label': ' Mountain/Wilderness Rescue', 'value': 'Mountain'},
                {'label': ' Disaster/Indiviual Tracking', 'value': 'Disaster'},
                {'label': ' Reset (All Records)', 'value': 'Reset'}
            ],
            value='Reset',
            labelStyle={'display': 'inline-block', 'marginRight': '20px', 'fontSize': '14px'}
        )
    ], style={'padding': '20px'}),
        
#FIXME Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.
    
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
#FIXME: Set up the features for your interactive data table to make it user-friendly for your client
#If you completed the Module Six Assignment, you can copy in the code you created here 
                        
        #Client ease-of-use:
        page_size=5,
        sort_action="native",
        filter_action="native",
        
        #Used for map callback functions
        row_selectable="single",
        selected_rows=[0],
                         
        #Used to further stylize the data table
        style_table={'overflow': 'auto'},
        style_cell={'textAlign': 'left', 'padding': '5px', 'fontFamily': 'Arial'},
        style_header={'backgroundColor': '#f4f4f4', 'fontWeight': 'bold'},                        
    ),
                        
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
            style={'display' : 'flex'},
            children=[
                html.Div(
                    id='map-id',
                    className='col s12 m6',
                    style={'flex': '1', 'padding': '10px', 'overflow': 'hidden'}
                ),
                html.Div(
                    id='graph-id',
                    className='col s12 m6',
                    style={'flex': '1', 'padding': '10px', 'overflow': 'hidden'}
            )
        ])
])
        
#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    if filter_type == 'Water':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
        
    elif filter_type == 'Mountain':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    elif filter_type == 'Disaster':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }
        
    else:
        query = {}
        
    results = db.read(query)
    filtered_df = pd.DataFrame(list(results))
    
    if filtered_df.empty:
        return []
    
    if '_id' in filtered_df.columns:
        filtered_df.drop(columns=['_id'], inplace=True)
        
    # Example: return your filtered data back to the datatable
    #df = pd.DataFrame.from_records(db.read(query))
    return filtered_df.to_dict('records')
                
## FIX ME Add code to filter interactive data table with MongoDB queries
#
#        
#        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
#        data=df.to_dict('records')
#       
#       
#        return (data,columns)

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('filter-type', 'value'),
    Input('datatable-id', "derived_virtual_data")])
def update_graphs(filter_type, viewData):
    
    # Prevents graph from breaking down on an empty dataframe
    if filter_type == 'Reset':
        return None
    
    #Strucural Validation
    if viewData is None or not isinstance(viewData, list) or len(viewData) == 0:
        return dash.no_update
    try:        
        df_local = pd.DataFrame(viewData)
        
        if 'breed' not in df_local.columns:
            return None
                
        breed_counts = df_local['breed'].value_counts().reset_index()
        breed_counts.columns = ['Breed', 'Count']
                             
        chart_fig = px.pie(
            breed_counts,
            values='Count',
            names='Breed',
            hole=0.4,
            title="Preferred Rescue Breed Demographics",
            color_discrete_sequence=px.colors.sequential.RdBu
        )
    
        chart_fig.update_layout(
            width=450,
            height=400,
            margin={"r":10,"t":40,"l":10,"b":10},
            font=dict(size=10),
            showlegend=True
        )
        chart_fig.update_layout(margin={"r":20,"t":40,"l":20,"b":20})
                             
        return dcc.Graph(figure=chart_fig, config={'displayModeBar': False})
                
    except Exception:
        return dash.no_update
    
    ###FIX ME ####
    # add code for chart of your choice (e.g. pie chart) #

    #return [
    #    dcc.Graph(            
    #        figure = px.pie(df, names='breed', title='Preferred Animals')
    #    )    
    #]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if not selected_columns:
        return []
    
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, selected_rows):  
    if viewData is None or len(viewData) == 0:
        return html.Div("No map data available")
    
    dff = pd.DataFrame.from_dict(viewData)
    
    if isinstance(selected_rows, list) and len(selected_rows) > 0:
        row = selected_rows[0]
    else:
        row = 0
    # Because we only allow single row selection, the list can be converted to a row index here
    if row >= len(dff) or row < 0:
        row = 0
        
    try:
        lat = float(dff.iloc[row]['location_lat'])
        lon = float(dff.iloc[row]['location_long'])
        breed = dff.iloc[row]['breed']
        name = dff.iloc[row].get('name', 'Unnamed')
    except (KeyError, IndexError):
        lat, lon, breed, name = 30.75, -97.48, "Unknown", "N/A"
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[lat,lon], zoom=12, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(f"Name: {name} | Breed: {breed}"),
                dl.Popup([
                    html.H1(f"Animal Name: {name}"),
                    html.P(f"Breed: {breed}"),
                    html.P(f"Coordinates: {lat}, {lon}")
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Successfully connected and authenticated to MongoDb!
Dash app running on https://politiccinema-unitlinda-3000.codio.io/proxy/8050/
